In [1]:
# Cell 1: GPU check and Google Drive mount
import torch
from google.colab import drive

print("GPU available:", torch.cuda.is_available())
print("Name:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Cell 2: Library installation
# rasterio reads GeoTIFF files; segmentation-models-pytorch provides the U-Net architecture
!pip install -q rasterio segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.6 MB/s eta 0:00:00


In [12]:
# Cell 3: Repository update and package import path
import sys
from pathlib import Path

REPO_URL = "https://github.com/Prhm93/floodfm-sen1floods11.git"
REPO_DIR = Path("/content/floodfm-sen1floods11")

# Clone on the first run in a session; pull the latest commits on later runs
if REPO_DIR.exists():
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

# The src folder is added to the Python search path, so that "import floodfm" finds the package
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.63 KiB | 1.63 MiB/s, done.
From https://github.com/Prhm93/floodfm-sen1floods11
   198675c..dfb7576  main       -> origin/main
Updating 198675c..dfb7576
Fast-forward
 src/floodfm/evaluate.py | 76 +++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 76 insertions(+)
 create mode 100644 src/floodfm/evaluate.py


In [4]:
# Cell 4: Dataset copy from Google Drive to the local Colab disk
import shutil
import time

DRIVE_ROOT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/data/sen1floods11")
LOCAL_ROOT = Path("/content/data/sen1floods11")
STATS_PATH = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/results/norm_stats.json")
FOLDERS = ["HandLabeled/S1Hand", "HandLabeled/S2Hand", "HandLabeled/LabelHand", "splits/flood_handlabeled"]

# Copy only if the local copy is not complete (446 chips in each layer)
def local_copy_complete():
    """Returns True if each of the three layer folders on the local disk contains 446 .tif files."""
    return all(len(list((LOCAL_ROOT / f).glob("*.tif"))) == 446 for f in FOLDERS[:3])

if local_copy_complete():
    print("Local copy already complete.")
else:
    t0 = time.time()
    for folder in FOLDERS:
        shutil.copytree(DRIVE_ROOT / folder, LOCAL_ROOT / folder, dirs_exist_ok=True)
    print(f"Copy time: {time.time() - t0:.0f} s")

# File counts on the local disk
for folder in FOLDERS:
    print(f"{folder}: {len(list((LOCAL_ROOT / folder).iterdir()))} files")

Copy time: 335 s
HandLabeled/S1Hand: 446 files
HandLabeled/S2Hand: 446 files
HandLabeled/LabelHand: 446 files
splits/flood_handlabeled: 4 files


In [5]:
# Cell 4a: Read time of 20 chips from Google Drive and from the local disk
from floodfm.data import read_split, read_chip

chip_ids = read_split(LOCAL_ROOT, "train")[:20]

for name, root in [("Drive", DRIVE_ROOT), ("Local disk", LOCAL_ROOT)]:
    t0 = time.time()
    for chip_id in chip_ids:
        read_chip(root, chip_id)
    print(f"{name}: {time.time() - t0:.1f} s for 20 chips")

Drive: 1.0 s for 20 chips
Local disk: 0.5 s for 20 chips


In [6]:
# Cell 5: GPU memory and speed test (6 training steps for each input type)
import time
import torch.nn as nn
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss
from torch.utils.data import DataLoader
from floodfm.dataset import Sen1Floods11

device = torch.device("cuda")
BATCH_SIZE = 8
IN_CHANNELS = {"s1": 2, "s2": 13}

# Loss: cross-entropy plus Dice; label -1 (no data) is ignored by both
ce_loss = nn.CrossEntropyLoss(ignore_index=-1)
dice_loss = DiceLoss(mode="multiclass", ignore_index=-1)

for modality in ["s1", "s2"]:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # U-Net with a ResNet-34 encoder, random start (no pretrained weights), 2 output classes
    model = smp.Unet(encoder_name="resnet34", encoder_weights=None,
                     in_channels=IN_CHANNELS[modality], classes=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler("cuda")

    dataset = Sen1Floods11(LOCAL_ROOT, "train", modality, STATS_PATH, augment=True)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

    # Six training steps with mixed precision; each step time includes data loading
    step_times = []
    t0 = time.time()
    for step, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(images)
        loss = ce_loss(logits.float(), labels) + dice_loss(logits.float(), labels)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()
        step_times.append(time.time() - t0)
        t0 = time.time()
        if step == 5:
            break

    # Mean step time without the first (warm-up) step, and the estimated time of one epoch
    mean_step = sum(step_times[1:]) / len(step_times[1:])
    steps_per_epoch = len(dataset) // BATCH_SIZE
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    print(f"{modality}: loss after 6 steps = {loss.item():.3f}")
    print(f"    peak GPU memory = {peak_gb:.1f} GB")
    print(f"    mean step time = {mean_step:.2f} s, steps per epoch = {steps_per_epoch}, "
          f"estimated epoch time = {mean_step * steps_per_epoch:.0f} s")

    del model, optimizer, loader, dataset

s1: loss after 6 steps = 1.317
    peak GPU memory = 2.4 GB
    mean step time = 0.23 s, steps per epoch = 31, estimated epoch time = 7 s
s2: loss after 6 steps = 1.249
    peak GPU memory = 2.5 GB
    mean step time = 0.27 s, steps per epoch = 31, estimated epoch time = 8 s


In [10]:
# Cell 6: Short test run (3 epochs, radar, seed 0) in a separate test folder
import importlib
import floodfm.train
importlib.reload(floodfm.train)
from floodfm.train import train

TEST_ROOT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/checkpoints/_test")
history = train(LOCAL_ROOT, STATS_PATH, TEST_ROOT, modality="s1", seed=0, epochs=3)

epoch   1 | loss 0.6261 | valid IoU 0.2869 | best 0.2869 | 9.8 s
epoch   2 | loss 0.4768 | valid IoU 0.5826 | best 0.5826 | 9.6 s
epoch   3 | loss 0.4360 | valid IoU 0.6153 | best 0.6153 | 9.8 s


In [11]:
# Cell 7: Resume test: the 3-epoch test run continues to epoch 4 from its last checkpoint
history = train(LOCAL_ROOT, STATS_PATH, TEST_ROOT, modality="s1", seed=0, epochs=4)

Resumed after epoch 3 (best validation IoU so far: 0.615)
epoch   4 | loss 0.4199 | valid IoU 0.5984 | best 0.6153 | 9.7 s


In [13]:
# Cell 8: Six U-Net runs (2 input types x 3 seeds): training, then evaluation; safe to re-run after a disconnect
import subprocess
import floodfm.evaluate
importlib.reload(floodfm.train)
importlib.reload(floodfm.evaluate)
from floodfm.train import train
from floodfm.evaluate import evaluate_run

PROJECT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content")
CKPT_ROOT = PROJECT / "checkpoints"
RESULTS_DIR = PROJECT / "results"
PRED_DIR = PROJECT / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
EPOCHS = 100

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

for modality in ["s1", "s2"]:
    for seed in [0, 1, 2]:
        method = f"unet_{modality}_seed{seed}"
        results_path = RESULTS_DIR / f"{method}.json"
        if results_path.exists():
            print(f"{method}: results file exists, skipped")
            continue

        print(f"\n===== {method}: training =====")
        train(LOCAL_ROOT, STATS_PATH, CKPT_ROOT, modality=modality, seed=seed, epochs=EPOCHS)

        print(f"===== {method}: evaluation =====")
        res = evaluate_run(
            LOCAL_ROOT, STATS_PATH, CKPT_ROOT / method / "best.pt", modality, method,
            description=f"U-Net (ResNet-34 encoder, no pretrained weights) on {modality}, seed {seed}, "
                        f"{EPOCHS} epochs, best validation epoch; pixels without input predicted as not water",
            results_path=results_path, prediction_path=PRED_DIR / f"{method}_bolivia.npz", code_commit=commit,
        )
        for split in ["test", "bolivia"]:
            s = res["splits"][split]["overall"]
            print(f"{method} | {split}: IoU = {s['iou']:.3f}, F1 = {s['f1']:.3f}")


===== unet_s1_seed0: training =====
epoch   1 | loss 0.6261 | valid IoU 0.2869 | best 0.2869 | 9.8 s
epoch   2 | loss 0.4795 | valid IoU 0.5831 | best 0.5831 | 10.0 s
epoch   3 | loss 0.4522 | valid IoU 0.5662 | best 0.5831 | 9.9 s
epoch   4 | loss 0.4392 | valid IoU 0.5957 | best 0.5957 | 9.9 s
epoch   5 | loss 0.4351 | valid IoU 0.6053 | best 0.6053 | 9.9 s
epoch   6 | loss 0.4510 | valid IoU 0.5902 | best 0.6053 | 10.0 s
epoch   7 | loss 0.4115 | valid IoU 0.5796 | best 0.6053 | 9.9 s
epoch   8 | loss 0.4194 | valid IoU 0.6257 | best 0.6257 | 10.0 s
epoch   9 | loss 0.4138 | valid IoU 0.5469 | best 0.6257 | 10.0 s
epoch  10 | loss 0.4583 | valid IoU 0.5755 | best 0.6257 | 10.0 s
epoch  11 | loss 0.4552 | valid IoU 0.5891 | best 0.6257 | 10.1 s
epoch  12 | loss 0.4637 | valid IoU 0.5787 | best 0.6257 | 10.0 s
epoch  13 | loss 0.4511 | valid IoU 0.5770 | best 0.6257 | 10.0 s
epoch  14 | loss 0.4386 | valid IoU 0.5968 | best 0.6257 | 10.1 s
epoch  15 | loss 0.4421 | valid IoU 0.6027 |

In [14]:
# Cell 9: Summary of all results files (baselines and U-Net runs), saved as CSV
import json
import pandas as pd

def load_results(name):
    """Reads one results file from the Drive results folder."""
    return json.loads((RESULTS_DIR / f"{name}.json").read_text())

def overall(r, split, key):
    """Returns one overall score of one split from a results file."""
    return r["splits"][split]["overall"][key]

rows = []

# Baselines
for name, label in [("otsu_vh", "Otsu (VH)"), ("fixed_vh", "Fixed VH threshold")]:
    r = load_results(name)
    rows.append({"method": label, "seed": None, "best_epoch": None, "valid_iou": overall(r, "valid", "iou"),
                 "test_iou": overall(r, "test", "iou"), "test_f1": overall(r, "test", "f1"),
                 "bolivia_iou": overall(r, "bolivia", "iou"), "bolivia_f1": overall(r, "bolivia", "f1")})

# U-Net runs
for modality, label in [("s1", "U-Net radar (S1)"), ("s2", "U-Net optical (S2)")]:
    for seed in [0, 1, 2]:
        name = f"unet_{modality}_seed{seed}"
        if not (RESULTS_DIR / f"{name}.json").exists():
            print("Missing results file:", name)
            continue
        r = load_results(name)
        rows.append({"method": label, "seed": seed, "best_epoch": r["checkpoint_epoch"],
                     "valid_iou": overall(r, "valid", "iou"),
                     "test_iou": overall(r, "test", "iou"), "test_f1": overall(r, "test", "f1"),
                     "bolivia_iou": overall(r, "bolivia", "iou"), "bolivia_f1": overall(r, "bolivia", "f1")})

per_run = pd.DataFrame(rows)
per_run.to_csv(RESULTS_DIR / "stage3_per_run.csv", index=False)
print("Each run:")
print(per_run.round(3).to_string(index=False))

# Mean, minimum and maximum over the three seeds of each U-Net
unet_runs = per_run[per_run["seed"].notna()]
summary = unet_runs.groupby("method")[["valid_iou", "test_iou", "bolivia_iou"]].agg(["mean", "min", "max"]).round(3)
print("\nU-Net over 3 seeds:")
print(summary.to_string())

Each run:
            method  seed  best_epoch  valid_iou  test_iou  test_f1  bolivia_iou  bolivia_f1
         Otsu (VH)   NaN         NaN      0.220     0.230    0.374        0.403       0.575
Fixed VH threshold   NaN         NaN      0.492     0.524    0.688        0.531       0.694
  U-Net radar (S1)   0.0        56.0      0.648     0.663    0.798        0.688       0.815
  U-Net radar (S1)   1.0        64.0      0.643     0.673    0.805        0.676       0.807
  U-Net radar (S1)   2.0        66.0      0.648     0.665    0.799        0.507       0.672
U-Net optical (S2)   0.0       100.0      0.837     0.828    0.906        0.777       0.875
U-Net optical (S2)   1.0        88.0      0.830     0.816    0.899        0.740       0.850
U-Net optical (S2)   2.0        81.0      0.828     0.817    0.899        0.781       0.877

U-Net over 3 seeds:
                   valid_iou               test_iou               bolivia_iou              
                        mean    min    max     me

In [15]:
# Cell 10: Bolivia chip-level IoU for the fixed threshold and the three radar U-Net seeds
def chip_iou(c):
    """IoU of one chip; NaN when the chip has no water in the label and no predicted water."""
    union = c["tp"] + c["fp"] + c["fn"]
    return c["tp"] / union if union > 0 else float("nan")

fixed = load_results("fixed_vh")
radar = {f"seed{s}": load_results(f"unet_s1_seed{s}") for s in [0, 1, 2]}

rows = []
for chip_id, c in fixed["splits"]["bolivia"]["chips"].items():
    scored = c["tp"] + c["fp"] + c["fn"] + c["tn"]
    row = {"chip": chip_id, "scored_pixels": scored,
           "water_share": (c["tp"] + c["fn"]) / scored if scored else float("nan"),
           "fixed_iou": chip_iou(c)}
    for name, r in radar.items():
        row[f"{name}_iou"] = chip_iou(r["splits"]["bolivia"]["chips"][chip_id])
    rows.append(row)

chips = pd.DataFrame(rows).sort_values("scored_pixels", ascending=False)
print(chips.round(3).to_string(index=False))

# Total false alarms and missed water on Bolivia for each radar seed
print()
for name, r in radar.items():
    o = r["splits"]["bolivia"]["overall"]
    print(f"radar {name}: TP = {o['tp']:,}  FP = {o['fp']:,}  FN = {o['fn']:,}")

          chip  scored_pixels  water_share  fixed_iou  seed0_iou  seed1_iou  seed2_iou
Bolivia_379434         262144        0.041      0.125      0.231      0.264      0.258
Bolivia_195474         261335        0.006      0.010      0.007      0.008      0.014
Bolivia_360519         254091        0.011      0.017      0.020      0.023      0.034
Bolivia_129334         237814        0.659      0.700      0.878      0.874      0.668
 Bolivia_23014         209479        0.037      0.558      0.726      0.768      0.763
Bolivia_290290         204633        0.096      0.381      0.460      0.584      0.566
 Bolivia_60373         199019        0.063      0.398      0.732      0.730      0.733
 Bolivia_76104         190965        0.000      0.000      0.000      0.000      0.000
Bolivia_314919         188845        0.443      0.553      0.699      0.717      0.476
Bolivia_242570         175507        0.156      0.309      0.160      0.042      0.023
Bolivia_312675         158008        0.061 